In [1]:
!pip install -q -U bitsandbytes>=0.46.1 peft transformers datasets accelerate

In [ ]:


import os, sys, gc, time, math, json, random, warnings, argparse, platform, importlib, subprocess

_HF_TOKEN = os.environ.get("HF_TOKEN")
if _HF_TOKEN:
    from huggingface_hub import login
    login(_HF_TOKEN)
else:
    print("  (i) HF_TOKEN not set in the environment; proceeding without an explicit "
          "huggingface_hub login. Set HF_TOKEN if the model/dataset needs auth.")

try:
    import bitsandbytes, peft, datasets, accelerate, scipy              
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "bitsandbytes>=0.46.1", "peft>=0.14", "datasets", "accelerate", "scipy"],
        check=True,
    )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")


CONFIG = dict(
                               
                                       
    MODEL = "microsoft/phi-2",
    SEEDS=[42, 123, 456],

                                                                             
    RANKS=[4, 8, 16, 32, 64],

                                                                           
    ALPHA_SCHEDULE="2r",                                          
    ALPHA_FIXED_VALUE=16,                                                          

                                                                             
    USE_4BIT=True,                                                                    

                                                                             
    TARGET_SCOPE="attn",                                      

                                                                              
    RUN_REGULARIZATION_CONDITIONS=True,
    WD_STRONG=0.10,                                                                     
    DROPOUT_HIGH=0.20,
    NUCLEAR_NORM_COEF=0.0,                                                               

                                                                              
    TRAIN_STEPS=200,
    PROBE_STEP=50,                                                                     
    EARLY_LOG_STRIDE=5,                                                     
    EARLY_LOG_UNTIL=100,
    LATE_LOG_EVERY=50,                                                          

    TRAIN_SAMPLES=2000,
    VAL_SAMPLES=200,
    MAX_SEQ_LEN=256,
    BATCH_SIZE=4,
    GRAD_ACCUM=4,
    LEARNING_RATE=2e-4,
    WARMUP_RATIO=0.1,
    DATA_SEED=1234,                                                               

                                                                               
    LOSS_SCOPE="full_sequence",                                             

                                                                              
    EVAL_TRUTHFUL=300,
    EVAL_ARC=300,

                                                                               
    SEE_RANK_TAU_GRID=[0.10, 0.15, 0.20, 0.25, 0.30],
    SEE_RANK_DEFAULT_RANK=16,                                                 

                                                                               
    RUN_TRUNCATION_TEST=True,
    TRUNCATION_K_COUNT=None,                                                             
                                                                                                

                                                                             
    GPU_INDEX=0,                                                                 
                                                                                       
                                                                                          
                                                                                         
                                                                                      
                                                                                            
                                                                                           

    OUT_DIR="/kaggle/working/",
)

                                                                            
                                                                           
                                                                             
                                                                          
                                                                              
                                                                           
                                                                             
 
                                                                          
                                                                          
                                                                  
                                                                        
                                                                       
                 
                                                                            
FAST_MODE = True

if FAST_MODE:
    CONFIG.update(
        SEEDS=[42, 123],                                                                
        TRAIN_STEPS=40,
        PROBE_STEP=10,                                                                          
        EARLY_LOG_STRIDE=5,                                                             
        EARLY_LOG_UNTIL=20,
        LATE_LOG_EVERY=20,                                                              
        VAL_SAMPLES=40,                                                                          
        EVAL_TRUTHFUL=20,                                                                     
        EVAL_ARC=20,                             
        TRUNCATION_K_COUNT=2,                                       
    )

                                                                           
                                                                         
_parser = argparse.ArgumentParser(add_help=False)
_parser.add_argument("--model", type=str, default=None)
_parser.add_argument("--no-4bit", action="store_true")
_parser.add_argument("--target-scope", type=str, default=None, choices=["attn", "attn_mlp"])
_parser.add_argument("--fast", action="store_true", help="force FAST_MODE scale-down")
_parser.add_argument("--full", action="store_true", help="force full-scale (overrides FAST_MODE)")
_parser.add_argument("--gpu", type=int, default=None, help="physical GPU index to pin the run to")
_args, _ = _parser.parse_known_args()
if _args.model:
    CONFIG["MODEL"] = _args.model
if _args.no_4bit:
    CONFIG["USE_4BIT"] = False
if _args.target_scope:
    CONFIG["TARGET_SCOPE"] = _args.target_scope
if _args.gpu is not None:
    CONFIG["GPU_INDEX"] = _args.gpu
if _args.full:
                                                                       
    CONFIG.update(SEEDS=[42, 123, 456], TRAIN_STEPS=200, PROBE_STEP=50,
                  EARLY_LOG_STRIDE=5, EARLY_LOG_UNTIL=100, LATE_LOG_EVERY=50,
                  VAL_SAMPLES=200, EVAL_TRUTHFUL=300, EVAL_ARC=300,
                  TRUNCATION_K_COUNT=None)
elif _args.fast:
                                                                                  
    CONFIG.update(SEEDS=[42, 123], TRAIN_STEPS=40, PROBE_STEP=10,
                  EARLY_LOG_STRIDE=5, EARLY_LOG_UNTIL=20, LATE_LOG_EVERY=20,
                  VAL_SAMPLES=40, EVAL_TRUTHFUL=20, EVAL_ARC=20,
                  TRUNCATION_K_COUNT=2)

                                                                            
                                                                         
                                                                            
                                                                                    

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
    import torch.nn as nn
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import LambdaLR
    from torch.utils.data import DataLoader
except Exception as e:
    print(f"FATAL: torch import failed: {e}")
    sys.exit(1)

try:
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import get_peft_model, LoraConfig, TaskType
except Exception as e:
    print(f"FATAL: transformers/datasets/peft import failed: {e}")
    sys.exit(1)

DEVICE = f"cuda:{CONFIG['GPU_INDEX']}" if torch.cuda.is_available() else "cpu"
if DEVICE.startswith("cuda"):
    torch.cuda.set_device(CONFIG["GPU_INDEX"])                                                      
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)


def safe_print(*a, **kw):
    try:
        print(*a, **kw)
    except Exception:
        pass


safe_print(f"Device : {DEVICE}")
if DEVICE.startswith("cuda"):
    safe_print(f"GPU    : {torch.cuda.get_device_name(CONFIG['GPU_INDEX'])}  "
               f"(pinned to a single GPU; {torch.cuda.device_count()} visible)")
    safe_print(f"VRAM   : {torch.cuda.get_device_properties(CONFIG['GPU_INDEX']).total_memory/1024**3:.1f} GB")

                                                                            
                                                                        
                                                                            

_ATTN_LEAF_CANDIDATES = {
    "q_proj", "k_proj", "v_proj", "o_proj", "dense", "out_proj",
    "Wqkv", "c_attn", "c_proj", "query_key_value", "qkv_proj", "o_proj",
}
_MLP_LEAF_CANDIDATES = {
    "gate_proj", "up_proj", "down_proj", "fc1", "fc2",
    "dense_h_to_4h", "dense_4h_to_h", "w1", "w2", "w3",
}


def _peft_builtin_target_map():
    """Try every known import path for PEFT's model_type -> target-module map."""
    for modpath in ("peft.utils.constants", "peft.utils.other", "peft.utils"):
        try:
            mod = importlib.import_module(modpath)
            mapping = getattr(mod, "TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING", None)
            if mapping:
                return mapping
        except Exception:
            continue
    return {}


def _introspect_linear_leaves(model, candidates):
    found = set()
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear):
            leaf = name.split(".")[-1]
            if leaf in candidates:
                found.add(leaf)
    return sorted(found)


def resolve_target_modules(model, scope="attn"):
    
    model_type = getattr(model.config, "model_type", None)
    builtin = _peft_builtin_target_map()
    modules = list(builtin.get(model_type, []) or [])

    if scope == "attn" and modules:
        attn_only = [m for m in modules if m in _ATTN_LEAF_CANDIDATES]
        if attn_only:
            modules = attn_only
        else:
                                                                             
                                                                          
                                                                           
                      
            modules = []

    if not modules:
        modules = _introspect_linear_leaves(model, _ATTN_LEAF_CANDIDATES)

    if not modules:
                                                                          
        safe_print("  ⚠ Could not resolve attention target modules automatically; "
                    "falling back to a generic guess. VERIFY before trusting results.")
        modules = ["q_proj", "k_proj", "v_proj", "o_proj"]

    if scope == "attn_mlp":
        mlp = _introspect_linear_leaves(model, _MLP_LEAF_CANDIDATES)
                                                                         
                                                                       
        mlp = sorted(set(mlp) | {m for m in (builtin.get(model_type, []) or []) if m in _MLP_LEAF_CANDIDATES})
        modules = sorted(set(modules) | set(mlp))

    return modules


def count_adapted_leaves(model, target_modules):
   
    n = 0
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear) and name.split(".")[-1] in target_modules:
            n += 1
    return n


def alpha_for_rank(r, schedule, fixed_value):
    if schedule == "fixed":
        return fixed_value
    if schedule == "2r":
        return 2 * r
    if schedule == "sqrt":
        return max(1, round(4 * math.sqrt(r)))                                 
    raise ValueError(f"Unknown ALPHA_SCHEDULE: {schedule}")


                                                                            
                                                                     
                                                                            

@torch.no_grad()
def get_adapter_layers(model):
    """Return (short_name, key, module) for every LoRA-adapted layer."""
    out = []
    for name, mod in model.named_modules():
        if hasattr(mod, "lora_A") and hasattr(mod, "lora_B"):
            for key in mod.lora_A.keys():
                parts = name.replace("base_model.model.", "").split(".")
                short_name = ".".join(parts[-2:]) if len(parts) >= 2 else name
                out.append((short_name, key, mod))
    return out


@torch.no_grad()
def dora_merged_delta(mod, key):
    
    try:
        mag_dict = getattr(mod, "lora_magnitude_vector", None)
        if mag_dict is None or key not in mag_dict:
            return None
        magnitude = mag_dict[key]
        if hasattr(magnitude, "weight"):
            magnitude = magnitude.weight
        magnitude = magnitude.float().flatten()

        base_layer = mod.get_base_layer() if hasattr(mod, "get_base_layer") else mod.base_layer
        W0 = base_layer.weight.float()
        A = mod.lora_A[key].weight.float()
        B = mod.lora_B[key].weight.float()
        scaling = mod.scaling[key] if hasattr(mod, "scaling") else 1.0
        BA = (B @ A) * scaling

        directional = W0 + BA
        col_norm = directional.norm(dim=1, keepdim=True).clamp_min(1e-8)                       
        W_new = directional * (magnitude.unsqueeze(1) / col_norm)
        return (W_new - W0)
    except Exception:
        return None


@torch.no_grad()
def spectral_snapshot(model, nominal_rank, step, is_dora=False):
    
    layer_rows, svd_rows = [], []
    for short_name, key, mod in get_adapter_layers(model):
        A = mod.lora_A[key].weight
        B = mod.lora_B[key].weight
        scaling = mod.scaling[key] if hasattr(mod, "scaling") else 1.0
        dW_ba = (B @ A).float() * scaling
        frob_sq_ba = torch.sum(dW_ba ** 2).item()
        degenerate = frob_sq_ba < 1e-16

        def _metrics_from(dW, frob_sq, delta_type):
            if frob_sq < 1e-16:
                return {
                    "step": step, "layer": short_name, "delta_type": delta_type,
                    "stable_rank": float("nan"), "top1_energy": float("nan"),
                    "rank_util_pct": float("nan"), "frob_norm": 0.0,
                    "degenerate": True,
                    "A_fro": torch.linalg.norm(A).item(),
                    "B_fro": torch.linalg.norm(B).item(),
                }, None
            S = torch.linalg.svdvals(dW)                                 
            S_sq = S ** 2
            total_e = S_sq.sum().item()
            sr = total_e / (S_sq[0].item() + 1e-16)                         
            row = {
                "step": step, "layer": short_name, "delta_type": delta_type,
                "stable_rank": round(sr, 6),
                "top1_energy": round(S_sq[0].item() / (total_e + 1e-16), 6),                    
                "rank_util_pct": round((sr / nominal_rank) * 100.0, 4),                          
                "frob_norm": round(math.sqrt(total_e), 6),
                "degenerate": False,
                "A_fro": torch.linalg.norm(A).item(),
                "B_fro": torch.linalg.norm(B).item(),
            }
            sv_rows = None
            if step == 0 or step >= CONFIG["TRAIN_STEPS"]:
                sv_rows = [{"step": step, "layer": short_name, "delta_type": delta_type,
                            "sv_index": i, "sv_value": round(v, 8)}
                           for i, v in enumerate(S[:nominal_rank].tolist())]
            return row, sv_rows

        row_ba, sv_ba = _metrics_from(dW_ba, frob_sq_ba, "BA")
        layer_rows.append(row_ba)
        if sv_ba:
            svd_rows.extend(sv_ba)

        if is_dora:
            dW_dora = dora_merged_delta(mod, key)
            if dW_dora is not None:
                frob_sq_dora = torch.sum(dW_dora ** 2).item()
                row_dora, sv_dora = _metrics_from(dW_dora, frob_sq_dora, "DoRA_merged")
                layer_rows.append(row_dora)
                if sv_dora:
                    svd_rows.extend(sv_dora)

    return layer_rows, svd_rows


@torch.no_grad()
def effective_rank_truncation_test(model, val_ds, nominal_rank, mean_sr, eval_ppl_fn):
    
    layers = get_adapter_layers(model)
    if not layers:
        return []
    ks = sorted(set(
        int(k) for k in
        [1, 2, max(1, math.ceil(mean_sr)) if not math.isnan(mean_sr) else 2,
         max(1, nominal_rank // 2), nominal_rank]
        if 1 <= k <= nominal_rank
    ))
    n_want = CONFIG.get("TRUNCATION_K_COUNT")
    if n_want and len(ks) > n_want:
                                                                                  
        if n_want == 1:
            ks = [ks[-1]]
        else:
            idx = np.linspace(0, len(ks) - 1, n_want).round().astype(int)
            ks = sorted(set(ks[i] for i in idx))

    originals = {}
    for short_name, key, mod in layers:
        originals[(short_name, key)] = (
            mod.lora_A[key].weight.data.clone(),
            mod.lora_B[key].weight.data.clone(),
        )

    rows = []
    try:
        for k in ks:
            for short_name, key, mod in layers:
                A0, B0 = originals[(short_name, key)]
                scaling = mod.scaling[key] if hasattr(mod, "scaling") else 1.0
                dW = (B0.float() @ A0.float()) * scaling
                U, S, Vh = torch.linalg.svd(dW, full_matrices=False)
                kk = min(k, S.shape[0])
                sqrtS = S[:kk].clamp_min(0).sqrt()
                Bk = U[:, :kk] * sqrtS
                Ak = sqrtS.unsqueeze(1) * Vh[:kk, :]
                denom = math.sqrt(scaling) if scaling > 0 else 1.0
                mod.lora_B[key].weight.data = (Bk / denom).to(B0.dtype)
                mod.lora_A[key].weight.data = (Ak / denom).to(A0.dtype)
            ppl_k = eval_ppl_fn(model, val_ds)
            rows.append({"k": k, "nominal_rank": nominal_rank, "val_ppl_at_k": ppl_k})
    finally:
        for short_name, key, mod in layers:
            A0, B0 = originals[(short_name, key)]
            mod.lora_A[key].weight.data = A0
            mod.lora_B[key].weight.data = B0

    return rows


                                                                            
                                                               
                                                                            

def load_data(tokenizer):
    safe_print("  Loading Alpaca-cleaned...")
    ds = load_dataset("yahma/alpaca-cleaned", split="train")

                                                               
    df = ds.to_pandas()
    n_before = len(df)
    df = df.drop_duplicates(subset=["instruction", "output"]).reset_index(drop=True)
    n_after = len(df)

    df = df.sample(frac=1.0, random_state=CONFIG["DATA_SEED"]).reset_index(drop=True)                         
    needed = CONFIG["TRAIN_SAMPLES"] + CONFIG["VAL_SAMPLES"]
    df = df.iloc[:min(needed, len(df))].reset_index(drop=True)

    train_df = df.iloc[:CONFIG["TRAIN_SAMPLES"]].reset_index(drop=True)
    val_df = df.iloc[CONFIG["TRAIN_SAMPLES"]:min(needed, len(df))].reset_index(drop=True)

    overlap = set(train_df["instruction"]).intersection(set(val_df["instruction"]))
    safe_print(f"  ✓ dedup removed {n_before - n_after} rows | "
               f"train={len(train_df)} val={len(val_df)} | instruction overlap={len(overlap)}")

    from datasets import Dataset
    tmpl = "### Instruction:\n{inst}\n\n### Response:\n{out}"

    def tok_fn(batch):
        insts, outs = batch["instruction"], batch["output"]
        texts = [tmpl.format(inst=i, out=o) for i, o in zip(insts, outs)]
        enc = tokenizer(texts, truncation=True, max_length=CONFIG["MAX_SEQ_LEN"],
                         padding="max_length", return_tensors="pt")
        labels = enc["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100

        if CONFIG["LOSS_SCOPE"] == "response_only":
                                                                               
            prefix_texts = ["### Instruction:\n{inst}\n\n### Response:\n".format(inst=i) for i in insts]
            prefix_lens = [len(tokenizer(p, truncation=True,
                                          max_length=CONFIG["MAX_SEQ_LEN"])["input_ids"])
                            for p in prefix_texts]
            for row_idx, plen in enumerate(prefix_lens):
                labels[row_idx, :min(plen, labels.shape[1])] = -100

        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}

    train_ds = Dataset.from_pandas(train_df).map(tok_fn, batched=True, batch_size=500,
                                                  remove_columns=train_df.columns.tolist())
    val_ds = Dataset.from_pandas(val_df).map(tok_fn, batched=True, batch_size=500,
                                              remove_columns=val_df.columns.tolist())
    train_ds.set_format("torch")
    val_ds.set_format("torch")

    manifest = {
        "loss_scope": CONFIG["LOSS_SCOPE"],
        "data_seed": CONFIG["DATA_SEED"],
        "n_dedup_removed": int(n_before - n_after),
        "n_train": len(train_ds), "n_val": len(val_ds),
        "instruction_overlap_train_val": len(overlap),
    }
    return train_ds, val_ds, manifest


                                                                            
                                                                             
                                                                            

@torch.no_grad()
def eval_ppl(model, val_ds):
    was_training = model.training
    model.eval()
    loader = DataLoader(val_ds, batch_size=8, shuffle=False)
    total_loss, total_tok = 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        n = (batch["labels"] != -100).sum().item()
        total_loss += out.loss.item() * n
        total_tok += n
    if was_training:
        model.train()
    return round(math.exp(min(total_loss / max(total_tok, 1), 20)), 4)


@torch.no_grad()
def _mc_eval(model, tokenizer, dataset_name, dataset_config, split, n_samples,
             question_key, choices_fn, correct_fn, prompt_fn, desc):
    try:
        ds = load_dataset(dataset_name, dataset_config, split=split)
    except Exception as e:
        safe_print(f"  ⚠ Could not load {dataset_name}/{dataset_config}: {e}")
        return None, []

    ds = ds.select(range(min(n_samples, len(ds))))
    was_training = model.training
    model.eval()
    per_item = []

    for item_idx, item in enumerate(tqdm(ds, desc=desc, leave=False, unit="q")):
        try:
            question = item[question_key]
            choices = choices_fn(item)
            gold = correct_fn(item)
            if gold is None or gold < 0 or gold >= len(choices):
                continue
        except Exception:
            continue

        prompts = [prompt_fn(question, c) for c in choices]
        inputs = tokenizer(prompts, return_tensors="pt", truncation=True,
                            max_length=96, padding=True).to(DEVICE)
        logits = model(**inputs).logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = inputs["input_ids"][:, 1:].contiguous()
        mask = inputs["attention_mask"][:, 1:].contiguous()

        tok_loss = nn.CrossEntropyLoss(reduction="none")(
            shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)
        ).view(shift_logits.size(0), -1)
        seq_loss = (tok_loss * mask.float()).sum(-1) / mask.sum(-1).clamp(min=1)

        pred = int(seq_loss.argmin())
        per_item.append({"item_index": item_idx, "correct": int(pred == gold)})

    if was_training:
        model.train()
    n_scored = len(per_item)
    n_correct = sum(r["correct"] for r in per_item)
    acc = n_correct / max(n_scored, 1)
    safe_print(f"  {desc}: {acc:.4f} ({n_correct}/{n_scored})")
    return round(acc, 4), per_item


def eval_truthfulqa(model, tokenizer):
    return _mc_eval(
        model, tokenizer, "truthful_qa", "multiple_choice", "validation", CONFIG["EVAL_TRUTHFUL"],
        "question", lambda it: it["mc1_targets"]["choices"],
        lambda it: it["mc1_targets"]["labels"].index(1) if 1 in it["mc1_targets"]["labels"] else None,
        lambda q, c: f"Q: {q}\nA: {c}", "TruthfulQA",
    )


def eval_arc(model, tokenizer):
    return _mc_eval(
        model, tokenizer, "allenai/ai2_arc", "ARC-Easy", "test", CONFIG["EVAL_ARC"],
        "question", lambda it: it["choices"]["text"],
        lambda it: it["choices"]["label"].index(it["answerKey"]) if it["answerKey"] in it["choices"]["label"] else None,
        lambda q, c: f"Question: {q}\nAnswer: {c}", "ARC-Easy",
    )


                                                                            
                                                                      
                                           
                                                                            

def cosine_schedule(optimizer, warmup, total):
    def lr_fn(step):
        if step < warmup:
            return step / max(1, warmup)
        p = (step - warmup) / max(1, total - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * p)))
    return LambdaLR(optimizer, lr_fn)


def logging_schedule():
    """M4/R6: dense early, coarser late. Always includes PROBE_STEP and final step."""
    steps = set(range(0, CONFIG["EARLY_LOG_UNTIL"] + 1, CONFIG["EARLY_LOG_STRIDE"]))
    steps |= set(range(CONFIG["EARLY_LOG_UNTIL"], CONFIG["TRAIN_STEPS"] + 1, CONFIG["LATE_LOG_EVERY"]))
    steps.add(CONFIG["PROBE_STEP"])
    steps.add(CONFIG["TRAIN_STEPS"])
    steps = sorted(s for s in steps if 0 <= s <= CONFIG["TRAIN_STEPS"])
    return steps


def nuclear_norm_penalty(model, coef):
    if coef <= 0:
        return torch.tensor(0.0, device=DEVICE)
    total = torch.tensor(0.0, device=DEVICE)
    for _, key, mod in get_adapter_layers(model):
        A = mod.lora_A[key].weight
        B = mod.lora_B[key].weight
        scaling = mod.scaling[key] if hasattr(mod, "scaling") else 1.0
        dW = (B @ A).float() * scaling
        total = total + torch.linalg.matrix_norm(dW, ord="nuc")
    return coef * total


def single_run(base_model, tokenizer, train_ds, val_ds, method_label, rank, alpha,
               dropout, wd, use_dora, seed, nuclear_coef=0.0):
    random.seed(seed); torch.manual_seed(seed); np.random.seed(seed)
    if DEVICE.startswith("cuda"):
        torch.cuda.manual_seed_all(seed)

    tag = f"{method_label}_s{seed}"
    safe_print(f"\n  ── {tag} {'─'*40}")

    peft_kwargs = dict(
        task_type=TaskType.CAUSAL_LM, r=rank, lora_alpha=alpha, lora_dropout=dropout,
        target_modules=CONFIG["_TARGET_MODULES"], bias="none", inference_mode=False,
    )
    if use_dora:
        try:
            peft_kwargs["use_dora"] = True
        except Exception:
            safe_print("     ⚠ DoRA not supported in this peft version, using LoRA")
            use_dora = False

    lora_cfg = LoraConfig(**peft_kwargs)
    model = get_peft_model(base_model, lora_cfg)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    safe_print(f"     method={method_label} rank={rank} alpha={alpha} params={n_par:,} "
               f"wd={wd} drop={dropout} dora={use_dora}")

    opt = AdamW(model.parameters(), lr=CONFIG["LEARNING_RATE"], weight_decay=wd)
    warmup = int(CONFIG["TRAIN_STEPS"] * CONFIG["WARMUP_RATIO"])
    sched = cosine_schedule(opt, warmup, CONFIG["TRAIN_STEPS"])
    loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, drop_last=True)

    log_steps = logging_schedule()

    spectral_rows, svd_rows = [], []
    snap, svd = spectral_snapshot(model, rank, 0, is_dora=use_dora)
    spectral_rows.extend(snap); svd_rows.extend(svd or [])

    model.train()
    step, losses, non_finite_events = 0, [], 0
    time_to_probe_sec = None
    t0 = time.time()
    opt.zero_grad()

    pbar = tqdm(total=CONFIG["TRAIN_STEPS"], desc=f"train[{tag}]", leave=False)
    while step < CONFIG["TRAIN_STEPS"]:
        for batch in loader:
            if step >= CONFIG["TRAIN_STEPS"]:
                break
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            try:
                with torch.amp.autocast("cuda", dtype=torch.float16, enabled=(DEVICE.startswith("cuda"))):
                    loss = model(**batch).loss
                    if nuclear_coef > 0:
                        loss = loss + nuclear_norm_penalty(model, nuclear_coef)
            except Exception:
                loss = model(**batch).loss

            loss_val = loss.item()
            if not math.isfinite(loss_val):
                non_finite_events += 1                                                  
            else:
                (loss / CONFIG["GRAD_ACCUM"]).backward()
                losses.append(loss_val)

            if (step + 1) % CONFIG["GRAD_ACCUM"] == 0:
                gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if not math.isfinite(float(gnorm)):
                    non_finite_events += 1
                opt.step(); sched.step(); opt.zero_grad()

            if step in log_steps:
                snap_t, svd_t = spectral_snapshot(model, rank, step, is_dora=use_dora)
                spectral_rows.extend(snap_t)
                svd_rows.extend(svd_t or [])

            if step == CONFIG["PROBE_STEP"]:
                time_to_probe_sec = time.time() - t0

            step += 1
            pbar.update(1)
    pbar.close()

    train_time = time.time() - t0
    if time_to_probe_sec is None:
        time_to_probe_sec = train_time

    snap, svd = spectral_snapshot(model, rank, CONFIG["TRAIN_STEPS"], is_dora=use_dora)
    spectral_rows.extend(snap); svd_rows.extend(svd or [])

    probe_val_ppl = None                                                           

    final_ppl = eval_ppl(model, val_ds)
    mc1, mc1_items = eval_truthfulqa(model, tokenizer)
    arc, arc_items = eval_arc(model, tokenizer)

    trunc_rows = []
    if CONFIG["RUN_TRUNCATION_TEST"]:
        ba_rows = [r for r in snap if r["delta_type"] == "BA" and not r["degenerate"]]
        mean_sr_final = float(np.mean([r["stable_rank"] for r in ba_rows])) if ba_rows else float("nan")
        try:
            trunc_rows = effective_rank_truncation_test(model, val_ds, rank, mean_sr_final, eval_ppl)
        except Exception as e:
            safe_print(f"     ⚠ truncation test failed: {e}")

    ba_final = [r for r in snap if r["delta_type"] == "BA" and not r["degenerate"]]
    result = {
        "model": CONFIG["_SHORT"], "method": method_label, "rank": rank, "alpha": alpha,
        "alpha_schedule": CONFIG["ALPHA_SCHEDULE"], "seed": seed,
        "dropout": dropout, "weight_decay": wd, "dora": use_dora,
        "use_4bit": CONFIG["USE_4BIT"], "target_scope": CONFIG["TARGET_SCOPE"],
        "loss_scope": CONFIG["LOSS_SCOPE"], "n_params": n_par,
        "final_ppl": final_ppl,
        "truthfulqa_mc1": mc1 if mc1 is not None else -1,
        "arc_easy": arc if arc is not None else -1,
        "n_truthfulqa_scored": len(mc1_items), "n_arc_scored": len(arc_items),
        "train_loss_last30": round(float(np.mean(losses[-30:])), 4) if losses else float("nan"),
        "non_finite_events": non_finite_events,
        "mean_sr_final": round(float(np.mean([r["stable_rank"] for r in ba_final])), 4) if ba_final else float("nan"),
        "min_sr_final": round(float(np.min([r["stable_rank"] for r in ba_final])), 4) if ba_final else float("nan"),
        "max_sr_final": round(float(np.max([r["stable_rank"] for r in ba_final])), 4) if ba_final else float("nan"),
        "mean_A_fro_final": round(float(np.mean([r["A_fro"] for r in ba_final])), 4) if ba_final else float("nan"),
        "mean_B_fro_final": round(float(np.mean([r["B_fro"] for r in ba_final])), 4) if ba_final else float("nan"),
        "train_min": round(train_time / 60, 2),
        "time_to_probe_min": round(time_to_probe_sec / 60, 2),
    }

    safe_print(f"     PPL={final_ppl:.2f}  MC1={mc1}  ARC={arc}  "
               f"mean_sr={result['mean_sr_final']}  non_finite={non_finite_events}  "
               f"{train_time/60:.1f} min")

    for r in mc1_items:
        r.update({"model": CONFIG["_SHORT"], "method": method_label, "seed": seed, "rank": rank, "task": "truthfulqa"})
    for r in arc_items:
        r.update({"model": CONFIG["_SHORT"], "method": method_label, "seed": seed, "rank": rank, "task": "arc_easy"})
    for r in trunc_rows:
        r.update({"model": CONFIG["_SHORT"], "method": method_label, "seed": seed})

    try:
        model.delete_adapter("default")
    except Exception:
        pass
    del model, opt, sched
    gc.collect()
    if DEVICE.startswith("cuda"):
        torch.cuda.empty_cache()

    return result, spectral_rows, svd_rows, mc1_items, arc_items, trunc_rows


                                                                            
                                                                       
                                                                            

def see_rank_select(ranks_sorted, sr_at_probe, tau):
    """Exactly the rule as specified: g_k = sr(r_k)/sr(r_k-1) - 1; stop (return
    r_{k-1}) the first time g_k < tau; else return the largest rank tried."""
    trace = []
    valid_ranks = [r for r in ranks_sorted if r in sr_at_probe and not math.isnan(sr_at_probe[r])]
    if len(valid_ranks) < 2:
        return (valid_ranks[0] if valid_ranks else None), trace
    for i in range(1, len(valid_ranks)):
        r_prev, r_cur = valid_ranks[i - 1], valid_ranks[i]
        sr_prev, sr_cur = sr_at_probe[r_prev], sr_at_probe[r_cur]
        g = (sr_cur / sr_prev - 1.0) if sr_prev > 0 else float("inf")
        trace.append({"r_prev": r_prev, "r_cur": r_cur, "gain": g, "tau": tau, "stop": g < tau})
        if g < tau:
            return r_prev, trace
    return valid_ranks[-1], trace


def run_see_rank_analysis(df_results, df_spectral):
    
    base_rows = df_results[df_results["method"].str.startswith("LoRA_r") &
                            ~df_results["method"].str.contains("wd|drop")].copy()
    if base_rows.empty:
        return pd.DataFrame(), pd.DataFrame()

    ranks_sorted = sorted(base_rows["rank"].unique().tolist())

    probe_spec = df_spectral[(df_spectral["step"] == CONFIG["PROBE_STEP"]) &
                              (df_spectral["delta_type"] == "BA") &
                              (~df_spectral["degenerate"])]
    sr_at_probe = probe_spec.groupby("rank")["stable_rank"].mean().to_dict()

    full_ppl = base_rows.groupby("rank")["final_ppl"].mean()
    best_rank_full_sweep = int(full_ppl.idxmin())

    probe_cost_total = base_rows.groupby("rank")["time_to_probe_min"].mean().sum()                                  
    completion_cost = base_rows.groupby("rank")["train_min"].mean() - base_rows.groupby("rank")["time_to_probe_min"].mean()
    full_sweep_cost_total = base_rows.groupby("rank")["train_min"].mean().sum()
    default_rank_cost = base_rows.groupby("rank")["train_min"].mean().get(CONFIG["SEE_RANK_DEFAULT_RANK"], float("nan"))

    rows, traces = [], []
    for tau in CONFIG["SEE_RANK_TAU_GRID"]:
        sel_rank, trace = see_rank_select(ranks_sorted, sr_at_probe, tau)
        for t in trace:
            t["tau"] = tau
            traces.append(t)
        end_to_end_cost = probe_cost_total + (completion_cost.get(sel_rank, float("nan")) if sel_rank else float("nan"))
        rows.append({
            "tau": tau,
            "selected_rank_see_rank": sel_rank,
            "selected_rank_val_loss_sweep": int(base_rows[base_rows["rank"].isin(ranks_sorted)]
                                                  .groupby("rank")["final_ppl"].mean().idxmin()),
            "default_rank_baseline": CONFIG["SEE_RANK_DEFAULT_RANK"],
            "best_rank_full_sweep": best_rank_full_sweep,
            "see_rank_matches_full_sweep": sel_rank == best_rank_full_sweep,
            "end_to_end_cost_min": round(end_to_end_cost, 2),
            "full_sweep_cost_min": round(full_sweep_cost_total, 2),
            "default_rank_cost_min": round(float(default_rank_cost), 2) if not math.isnan(default_rank_cost) else float("nan"),
        })

    return pd.DataFrame(rows), pd.DataFrame(traces)


                                                                            
                                                                        
                                                                            

def fit_functional_forms(ranks, srs):
    ranks = np.array(ranks, dtype=float)
    srs = np.array(srs, dtype=float)
    mask = np.isfinite(ranks) & np.isfinite(srs) & (srs > 0) & (ranks > 0)
    ranks, srs = ranks[mask], srs[mask]
    out = {}
    if len(ranks) < 3:
        return out

    def aic(resid, k_params):
        n = len(resid)
        rss = float(np.sum(resid ** 2))
        if rss <= 0 or n <= k_params:
            return float("inf")
        return n * math.log(rss / n) + 2 * k_params

                                               
    try:
        logr, logsr = np.log(ranks), np.log(srs)
        A = np.vstack([logr, np.ones_like(logr)]).T
        beta, loga = np.linalg.lstsq(A, logsr, rcond=None)[0]
        resid = logsr - (A @ [beta, loga])
        out["power_law"] = {"beta": float(beta), "a": float(math.exp(loga)), "aic": aic(resid, 2)}
    except Exception:
        pass

                                    
    try:
        logr = np.log(ranks)
        A = np.vstack([logr, np.ones_like(logr)]).T
        a, b = np.linalg.lstsq(A, srs, rcond=None)[0]
        resid = srs - (A @ [a, b])
        out["logarithmic"] = {"a": float(a), "b": float(b), "aic": aic(resid, 2)}
    except Exception:
        pass

                                                                         
    try:
        best = None
        for r0 in np.linspace(0.1, ranks.max() * 3, 200):
            x = ranks / (ranks + r0)
            S = float(np.dot(x, srs) / np.dot(x, x)) if np.dot(x, x) > 0 else 0.0
            resid = srs - S * x
            a = aic(resid, 2)
            if best is None or a < best[0]:
                best = (a, S, r0)
        if best:
            out["saturating"] = {"S": best[1], "r0": best[2], "aic": best[0]}
    except Exception:
        pass

    return out


                                                                            
                                                                      
                                                                            

def seed_level_correlation(df_results, x_col, y_col):
    from scipy import stats
    valid = df_results[(df_results[x_col].notna()) & (df_results[y_col] >= 0)]
    if len(valid) < 4:
        return None
    r, p = stats.pearsonr(valid[x_col], valid[y_col])
    return {"x": x_col, "y": y_col, "n": len(valid), "r": round(float(r), 4), "p": round(float(p), 6)}


def tost_equivalence(sample_a, sample_b, margin):
    from scipy import stats
    a, b = np.asarray(sample_a, dtype=float), np.asarray(sample_b, dtype=float)
    if len(a) < 2 or len(b) < 2:
        return None
    diff = a.mean() - b.mean()
    se = math.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    if se == 0:
        return {"diff": diff, "equivalent": abs(diff) < margin, "p_tost": float("nan")}
    t1 = (diff - (-margin)) / se
    t2 = (diff - margin) / se
    df = len(a) + len(b) - 2
    p1 = 1 - stats.t.cdf(t1, df)
    p2 = stats.t.cdf(t2, df)
    p_tost = max(p1, p2)
    return {"diff": round(float(diff), 4), "p_tost": round(float(p_tost), 6), "equivalent": bool(p_tost < 0.05)}


                                                                            
      
                                                                            

def build_methods(ranks):
    
    methods = []
    for r in ranks:
        a = alpha_for_rank(r, CONFIG["ALPHA_SCHEDULE"], CONFIG["ALPHA_FIXED_VALUE"])
        methods.append((f"LoRA_r{r}", r, a, 0.05, 0.0, False))

    if CONFIG["RUN_REGULARIZATION_CONDITIONS"]:
        mid_rank = ranks[len(ranks) // 2]
        a_mid = alpha_for_rank(mid_rank, CONFIG["ALPHA_SCHEDULE"], CONFIG["ALPHA_FIXED_VALUE"])
        methods.append((f"LoRA_r{mid_rank}_wd", mid_rank, a_mid, 0.05, CONFIG["WD_STRONG"], False))
        methods.append((f"LoRA_r{mid_rank}_drop", mid_rank, a_mid, CONFIG["DROPOUT_HIGH"], 0.0, False))
        methods.append((f"DoRA_r{mid_rank}", mid_rank, a_mid, 0.05, 0.0, True))

    return methods


def main():
    t_start = time.time()
    MODEL = CONFIG["MODEL"]
    CONFIG["_SHORT"] = MODEL.split("/")[-1]

    tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    safe_print(f"\nLoading {MODEL}  (4bit={CONFIG['USE_4BIT']}, gpu={CONFIG['GPU_INDEX']})...")
                                                                                    
                                                                    
                                                                                    
                                                                                 
    single_gpu_map = {"": CONFIG["GPU_INDEX"]} if DEVICE.startswith("cuda") else None
    load_kwargs = dict(device_map=single_gpu_map, trust_remote_code=True)
    if CONFIG["USE_4BIT"]:
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
    else:
        load_kwargs["torch_dtype"] = torch.float16

    base = AutoModelForCausalLM.from_pretrained(MODEL, **load_kwargs)
    base.config.use_cache = False
    safe_print("✓ Model loaded")

    CONFIG["_TARGET_MODULES"] = resolve_target_modules(base, CONFIG["TARGET_SCOPE"])
    n_adapted = count_adapted_leaves(base, CONFIG["_TARGET_MODULES"])
    safe_print(f"✓ Target modules (auto-resolved, model_type={getattr(base.config,'model_type',None)}, "
               f"scope={CONFIG['TARGET_SCOPE']}): {CONFIG['_TARGET_MODULES']}")
    safe_print(f"✓ This will attach LoRA to {n_adapted} Linear leaves per run "
               f"(PERF-2 sanity check — if this looks too high for the requested "
               f"scope, TARGET_SCOPE resolution needs another look before a long run).")

    train_ds, val_ds, data_manifest = load_data(tok)

    methods = build_methods(CONFIG["RANKS"])
    total_runs = len(methods) * len(CONFIG["SEEDS"])
    safe_print(f"\n{len(methods)} methods × {len(CONFIG['SEEDS'])} seeds = {total_runs} runs")

    all_results, all_spectral, all_svd, all_mc1, all_arc, all_trunc = [], [], [], [], [], []

    f_res = os.path.join(CONFIG["OUT_DIR"], f"spectral_{CONFIG['_SHORT']}_results.csv")
    f_traj = os.path.join(CONFIG["OUT_DIR"], f"spectral_{CONFIG['_SHORT']}_trajectory.csv")
    f_svd = os.path.join(CONFIG["OUT_DIR"], f"spectral_{CONFIG['_SHORT']}_svd.csv")
    f_items = os.path.join(CONFIG["OUT_DIR"], f"spectral_{CONFIG['_SHORT']}_downstream_items.csv")
    f_trunc = os.path.join(CONFIG["OUT_DIR"], f"spectral_{CONFIG['_SHORT']}_truncation.csv")
    f_seerank = os.path.join(CONFIG["OUT_DIR"], f"spectral_{CONFIG['_SHORT']}_see_rank.csv")
    f_seerank_trace = os.path.join(CONFIG["OUT_DIR"], f"spectral_{CONFIG['_SHORT']}_see_rank_trace.csv")
    f_forms = os.path.join(CONFIG["OUT_DIR"], f"spectral_{CONFIG['_SHORT']}_functional_forms.json")
    f_meta = os.path.join(CONFIG["OUT_DIR"], f"spectral_{CONFIG['_SHORT']}_metadata.json")

    run_bar = tqdm(total=total_runs, desc="sweep", unit="run")
    idx = 0
    for method_label, rank, alpha, dropout, wd, use_dora in methods:
        for seed in CONFIG["SEEDS"]:
            idx += 1
            run_bar.set_postfix_str(f"{method_label} s={seed}")
            try:
                res, spec, svd, mc1_items, arc_items, trunc_rows = single_run(
                    base, tok, train_ds, val_ds, method_label, rank, alpha,
                    dropout, wd, use_dora, seed, nuclear_coef=CONFIG["NUCLEAR_NORM_COEF"],
                )
                all_results.append(res)
                for r in spec:
                    r.update({"model": CONFIG["_SHORT"], "method": method_label, "seed": seed, "rank": rank})
                all_spectral.extend(spec)
                for r in svd:
                    r.update({"model": CONFIG["_SHORT"], "method": method_label, "seed": seed, "rank": rank})
                all_svd.extend(svd)
                all_mc1.extend(mc1_items)
                all_arc.extend(arc_items)
                all_trunc.extend(trunc_rows)

                pd.DataFrame(all_results).to_csv(f_res, index=False)
                pd.DataFrame(all_spectral).to_csv(f_traj, index=False)
            except Exception as e:
                safe_print(f"  ❌ FAILED {method_label} seed={seed}: {e}")
                import traceback; traceback.print_exc()
                gc.collect()
                if DEVICE.startswith("cuda"):
                    torch.cuda.empty_cache()
            run_bar.update(1)
    run_bar.close()

    df_res = pd.DataFrame(all_results)
    df_spec = pd.DataFrame(all_spectral)
    df_svd = pd.DataFrame(all_svd)
    df_items = pd.DataFrame(all_mc1 + all_arc)
    df_trunc = pd.DataFrame(all_trunc)

    df_res.to_csv(f_res, index=False)
    df_spec.to_csv(f_traj, index=False)
    if len(df_svd):
        df_svd.to_csv(f_svd, index=False)
    if len(df_items):
        df_items.to_csv(f_items, index=False)
    if len(df_trunc):
        df_trunc.to_csv(f_trunc, index=False)

    safe_print(f"\n✓ {f_res} ({len(df_res)} rows)")
    safe_print(f"✓ {f_traj} ({len(df_spec)} rows)")
    if len(df_svd): safe_print(f"✓ {f_svd} ({len(df_svd)} rows)")
    if len(df_items): safe_print(f"✓ {f_items} ({len(df_items)} rows)")
    if len(df_trunc): safe_print(f"✓ {f_trunc} ({len(df_trunc)} rows)")

                                                                               
    if len(df_res) and len(df_spec):
        try:
            df_see, df_trace = run_see_rank_analysis(df_res, df_spec)
            if len(df_see):
                df_see.to_csv(f_seerank, index=False)
                df_trace.to_csv(f_seerank_trace, index=False)
                safe_print(f"✓ {f_seerank} ({len(df_see)} rows)")
        except Exception as e:
            safe_print(f"  ⚠ SEE-Rank analysis failed: {e}")

                                                                                
    forms_out = {}
    if len(df_spec):
        final_ba = df_spec[(df_spec["step"] == CONFIG["TRAIN_STEPS"]) &
                            (df_spec["delta_type"] == "BA") & (~df_spec["degenerate"])]
        rank_sr = final_ba.groupby("rank")["stable_rank"].mean()
        if len(rank_sr) >= 3:
            forms_out = fit_functional_forms(rank_sr.index.tolist(), rank_sr.values.tolist())
            safe_print("\nFunctional-form fit (lower AIC = better):")
            for name, d in forms_out.items():
                safe_print(f"  {name}: {d}")
    with open(f_forms, "w") as fh:
        json.dump(forms_out, fh, indent=2)

                                                                                
    corr_summary = []
    if len(df_res) >= 4:
        for metric in ["final_ppl", "truthfulqa_mc1", "arc_easy"]:
            c = seed_level_correlation(df_res, "mean_sr_final", metric)
            if c:
                corr_summary.append(c)
        safe_print("\nSeed-level (not cell-mean) correlations vs mean_sr_final:")
        for c in corr_summary:
            safe_print(f"  {c}")

                                                                                 
    meta = {
        "config": {k: v for k, v in CONFIG.items() if not k.startswith("_")},
        "resolved_target_modules": CONFIG["_TARGET_MODULES"],
        "n_adapted_leaves": n_adapted,
        "model_config_summary": {
            k: getattr(base.config, k, None)
            for k in ["model_type", "hidden_size", "num_hidden_layers",
                      "num_attention_heads", "num_key_value_heads"]
        },
        "data_manifest": data_manifest,
        "seed_level_correlations": corr_summary,
        "library_versions": {
            "python": platform.python_version(),
            "torch": getattr(torch, "__version__", None),
        },
        "device": DEVICE,
        "total_wallclock_min": round((time.time() - t_start) / 60, 2),
    }
    for lib in ["transformers", "peft", "datasets", "bitsandbytes", "accelerate", "scipy"]:
        try:
            meta["library_versions"][lib] = importlib.import_module(lib).__version__
        except Exception:
            meta["library_versions"][lib] = "not available"

    with open(f_meta, "w") as fh:
        json.dump(meta, fh, indent=2, default=str)
    safe_print(f"✓ {f_meta}")

    total_time = (time.time() - t_start) / 60
    safe_print(f"\nDONE in {total_time:.1f} min")
    return df_res, df_spec, df_svd, df_items, df_trunc


if __name__ == "__main__":
    main()

  (i) HF_TOKEN not set in the environment; proceeding without an explicit huggingface_hub login. Set HF_TOKEN if the model/dataset needs auth.


Device : cuda:0
GPU    : Tesla T4  (pinned to a single GPU; 2 visible)
VRAM   : 14.6 GB


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

train[DoRA_r16_s123]:   0%|          | 0/40 [00:00<?, ?it/s]

TruthfulQA:   0%|          | 0/20 [00:00<?, ?q/s]

  TruthfulQA: 0.3000 (6/20)


ARC-Easy:   0%|          | 0/20 [00:00<?, ?q/s]

  ARC-Easy: 0.7500 (15/20)
     PPL=3.52  MC1=0.3  ARC=0.75  mean_sr=3.5882  non_finite=0  13.4 min

✓ /kaggle/working/spectral_phi-2_results.csv (16 rows)
✓ /kaggle/working/spectral_phi-2_trajectory.csv (7168 rows)
✓ /kaggle/working/spectral_phi-2_svd.csv (22016 rows)
✓ /kaggle/working/spectral_phi-2_downstream_items.csv (640 rows)
✓ /kaggle/working/spectral_phi-2_truncation.csv (32 rows)
✓ /kaggle/working/spectral_phi-2_see_rank.csv (5 rows)

Functional-form fit (lower AIC = better):
  power_law: {'beta': 0.24442346462911718, 'a': 1.748379115160747, 'aic': -24.347439897970972}
  logarithmic: {'a': 0.8153823482693549, 'b': 1.2830638363281215, 'aic': -17.484949842608103}
  saturating: {'S': 4.871251954363149, 'r0': np.float64(4.921608040201005), 'aic': -22.50091419070145}

Seed-level (not cell-mean) correlations vs mean_sr_final:
  {'x': 'mean_sr_final', 'y': 'final_ppl', 'n': 16, 'r': -0.0453, 'p': 0.867552}
  {'x': 'mean_sr_final', 'y': 'truthfulqa_mc1', 'n': 16, 'r': 0.2531, 'p': 0.